Creating a simple dataset.

In [11]:
%%writefile weblogs.txt
# Date, Time, IP, Method, URL, Status, ResponseSize
2025-10-11,09:15:10,10.0.0.2,GET,/home.html,200,1100
2025-10-11,09:15:14,10.0.0.3,GET,/shop.html,200,900
2025-10-11,09:15:18,10.0.0.4,GET,/support.html,404,480
2025-10-11,09:15:22,10.0.0.5,POST,/payment,500,140
2025-10-11,09:15:26,10.0.0.6,GET,/home.html,200,1100
2025-10-11,09:15:30,10.0.0.7,GET,/assets/logo.svg,200,300
2025-10-11,09:15:34,10.0.0.8,GET,/company.html,404,500
2025-10-11,09:15:39,10.0.0.9,POST,/signin,401,70
2025-10-11,09:15:44,10.0.0.10,GET,/home.html,200,1100
2025-10-11,09:15:49,10.0.0.11,POST,/payment,500,140
2025-10-11,09:15:53,10.0.0.12,GET,/support.html,404,480
2025-10-11,09:15:57,10.0.0.13,GET,/home.html,200,1100
2025-10-11,09:16:01,10.0.0.14,GET,/shop.html,200,900
2025-10-11,09:16:05,10.0.0.15,GET,/company.html,404,500
2025-10-11,09:16:09,10.0.0.16,POST,/payment,500,140
2025-10-11,09:16:13,10.0.0.17,GET,/assets/logo.svg,200,300
2025-10-11,09:16:17,10.0.0.18,GET,/support.html,404,480
2025-10-11,09:16:22,10.0.0.19,POST,/signin,401,70
2025-10-11,09:16:27,10.0.0.20,GET,/home.html,200,1100
2025-10-11,09:16:32,10.0.0.21,GET,/shop.html,200,900



Overwriting weblogs.txt


Mapper

In [13]:
from collections import defaultdict

# Mapper function
def mapper(line):
    fields = line.strip().split(',')
    if len(fields) != 7 or fields[0].startswith('#'):
        return []
    status = int(fields[5])   # Status code
    return [(status, 1)]


Shuffle و Reducer

In [14]:
# Shuffle / Grouping function
def shuffle(mapped_data):
    grouped = defaultdict(list)
    for key, value in mapped_data:
        grouped[key].append(value)
    return grouped

# Reducer function
def reducer(shuffled_data):
    reduced_output = {}
    for key, values_list in shuffled_data.items():
        reduced_output[key] = sum(values_list)
    return reduced_output



التنفيذ + Bonus (الأخطاء فقط)

In [15]:
mapped = []
with open("weblogs.txt", "r") as f:
    for line in f:
        mapped.extend(mapper(line))

shuffled = shuffle(mapped)
reduced = reducer(shuffled)

for code, count in sorted(reduced.items()):
    print(f"HTTP {code}: {count} requests")


HTTP 200: 10 requests
HTTP 401: 2 requests
HTTP 404: 5 requests
HTTP 500: 3 requests


(Bonus): حساب الأخطاء فقط

In [16]:
# Mapper for error requests only
def mapper_errors(line):
    fields = line.strip().split(',')
    if len(fields) != 7 or fields[0].startswith('#'):
        return []
    status = int(fields[5])
    if status < 400:
        return []   # تجاهل الطلبات الناجحة
    return [(status, 1)]

mapped_errors = []
with open("weblogs.txt", "r") as f:
    for line in f:
        mapped_errors.extend(mapper_errors(line))

shuffled_errors = shuffle(mapped_errors)
reduced_errors = reducer(shuffled_errors)

for status, count in sorted(reduced_errors.items()):
    print(f"HTTP {status} (error): {count} requests")


HTTP 401 (error): 2 requests
HTTP 404 (error): 5 requests
HTTP 500 (error): 3 requests
